In [ ]:
import vibechecker as vc
import sounddevice
import yaml
import pickle
from datetime import datetime, tzinfo
import pandas as pd
import numpy as np
import plotly.express as px
from path import Path
import h5py
import time
import os
import glob
import queue

In [ ]:
conf = vc.AcquisitionSettings(units='mm')
conf.binsize = 2
conf.maxfreq = 10000
simsensor = vc.VibeSensor.simulated()
vd = vc.DataCollector(config=conf)
vd.connect_sensor(simsensor)
sim = vd.stream

sim.source = (vc.GenerateTone,1,4000,0)
samp = vd.collect_sample()
vis = vd.visualize_init(samp)

In [ ]:
step = 100 # hz interval
tones = np.arange(step,conf.maxfreq,step)
psdv = []
rmsv = []
for tone in tones:
    sim.source = (vc.GenerateTone,1,tone,0)
    samp = vd.collect_sample()
    acc, rms = samp.get_accel(conf)
    fft, peaks = samp.fft(conf)
    psdv.append(fft.loc[(np.abs(fft.freq-tone)-1)<0].acc_0p.sum())
    rmsv.append(rms)

px.scatter(pd.DataFrame({'freq':tones, 'psd': psdv, 'rms':rmsv}), x='freq', y='psd')

In [ ]:
conf

In [ ]:
# Launch GUI
app = vc.GUI()
app.serve()

In [ ]:
# test load all datasets
datasets = list(map(Path,glob.glob('DEVDATA/*.h5')))
samps = []
for dataset in datasets:
    if dataset.basename().startswith('pytest'):
        continue

    samps.append(vc.VibeSample.load(dataset))


In [ ]:
# Test units comparisson
dataset = Path('DEVDATA/rotorkit_1800rpm_2025-12-30_15-03-02.h5')
samp = vc.VibeSample.load(dataset) # type: ignore

In [ ]:

config = vc.AcquisitionSettings(len(samp.data), samp.samplerate, units='g', integrate=False)
vtime = samp.time_vec
acc, rms = samp.get_accel(config)
df, peak = samp.fft(config)
df.psd.max()

In [ ]:
devs = vc.VibeSensor.find()

print('ID\tNAME')
for s in devs:
    print(f'{s.device_id}:\t{s.model_name}')

In [ ]:
q = queue.Queue()
def callback(sample):
    q.put(sample)

config = vc.AcquisitionSettings(vc.SAMPLERATES[2], vc.BLOCKSIZES[-1])
sensor = devs[-1]

# Close any existing stream if present and active
try:
    stream.close() # type: ignore
except Exception:
    pass

stream = sensor.connect(config, callback)

stream.start()
time.sleep(1)
stream.stop()
stream.close()

while q.qsize():
    sample = q.get()
q.shutdown()

ch = 0
vs = vc.VibeSample(sample['status'],
                   sample['timestamp'],
                   config.samplerate,
                   config.units,
                   sample['data'][:,ch])

td = np.arange(vs.blocksize) / vs.samplerate
df = pd.DataFrame({'time': td,
                   'accel0': sample['data'][:,0],
                   'accel1': sample['data'][:,1],
                   'accelm': np.mean(sample['data'], axis=1)})
px.line(df, x='time', y=['accel0', 'accel1', 'accelm'])

In [ ]:
# Generate and visualize simulated data
dev = vc.VibeSensor.find()

if len(dev)>1:
    sensor = dev[1]
else:
    sensor = dev[0]

print(sensor)

config = vc.AcquisitionSettings(4096,8000)

config.ensure_maxfreq(1000)
config.ensure_binsize(2.0)

vibr = vc.DataCollector(sensor=sensor, config=config)
samp = vibr.collect_sample()
vis = vibr.visualize_init(samp)

vibr.disconnect_sensor()